# RustPower Lesson 1: Building and Analyzing your First Grid ⚡️

Welcome! In this lesson we build a power grid from scratch, run a power flow, and analyze the results with Pandas.

### 🎯 Objectives
We will build a simple **110 kV two-bus system**:
1. **Bus 0 (Substation)** — slack bus held at 1.02 p.u. by an external grid.
2. **Bus 1 (Factory)** — a 30 MW industrial load.
3. **Line** — a 20 km cable connecting them.

Along the way you will meet the three core ideas of the API:
- **`grid.edit()`** — all topology changes happen inside a transaction.
- **`grid.solve()`** — always correct; it knows what needs rebuilding.
- **`grid.res_bus`** — results as DataFrames, mirroring pandapower.

In [1]:
import rustpower
import pandas as pd
import numpy as np

print(f"RustPower Core Ready! Version: {rustpower.version()}")

RustPower Core Ready! Version: 0.5.0-rc.2


## 1. Building with the transactional editor

Topology mutations go through `grid.edit()`. Inside the `with` block, commands are
buffered in a high-speed command queue and applied **once** on exit (fused insert).
If an exception is raised inside the block, the whole transaction is rolled back
and the grid is left untouched.

In [2]:
grid = rustpower.PowerGrid()
grid.set_base(sn_mva=100.0)

with grid.edit() as e:
    # 1. Substation: slack bus at 1.02 p.u.
    sub_id, sub_bus = e.add_bus(vn_kv=110.0, name="Substation")
    e.add_ext_grid(bus=sub_id, vm_pu=1.02)

    # 2. Factory: 30 MW / 10 MVar load
    fac_id, fac_bus = e.add_bus(vn_kv=110.0, name="Factory")
    factory_load = e.add_load(bus=fac_id, p_mw=30.0, q_mvar=10.0)

    # 3. Connect them
    e.add_line(from_bus=sub_id, to_bus=fac_id, length_km=20.0,
               r_ohm_per_km=0.1, x_ohm_per_km=0.1, max_i_ka=0.2)

print(f"Created grid with {grid.n_bus} buses and {grid.n_line} lines.")
grid.describe()

Created grid with 2 buses and 1 lines.


,element,count
0,bus,2
1,line,1
2,trafo,0
3,load,1
4,gen,0
5,ext_grid,1
6,shunt,0


## 2. Solving — no ceremony

There is no `init_pf()` to remember. The editor commit marked the topology dirty,
so `solve()` performs a full rebuild automatically and tells you it did so in the
returned `SolveReport` (which is truthy iff the solver converged).

In [3]:
report = grid.solve()
print(report)

if report:
    print(f"Converged in {report.iterations} iterations "
          f"({report.runtime_ms:.2f} ms, rebuild={report.rebuild!r})")

SolveReport(converged=true, iterations=2, runtime_ms=0.760, rebuild='full')
Converged in 2 iterations (0.76 ms, rebuild='full')


## 3. Results: DataFrames and element proxies

`grid.res_bus` / `grid.res_line` mirror pandapower's result tables. For single
elements, the handles returned during construction are **live proxies**: reading a
property queries the solver state directly.

In [4]:
print("--- Bus Results ---")
print(grid.res_bus)

print("\n--- Line Results ---")
print(grid.res_line)

# Element-level access through the proxy
print(f"\nFactory voltage: {fac_bus.vm_pu:.4f} p.u. at {fac_bus.va_degree:.3f} deg")
print(f"Minimum system voltage: {grid.res_bus['vm_pu'].min():.4f} p.u.")

--- Bus Results ---
   bus_id                v_pu     vm_pu  va_degree       p_mw     q_mvar
0       0  1.020000+0.000000j  1.020000   0.000000 -30.160924 -10.160924
1       1  1.013466-0.003241j  1.013471  -0.183226  30.000000  10.000000

--- Line Results ---
   from_bus  to_bus       p_mw     q_mvar      i_ka
0         0       1  30.160924  10.160924  0.283659

Factory voltage: 1.0135 p.u. at -0.183 deg
Minimum system voltage: 1.0135 p.u.


## 4. What-if studies: assign and solve

Parameter changes are plain property assignments — no transaction, no rebuild.
`solve()` takes the **incremental path** (only the injection vector is refreshed,
and the previous solution warm-starts the solver).

In [5]:
v_before = fac_bus.vm_pu

factory_load.p_mw = 45.0          # the factory expands!
report = grid.solve()

print(report)
print(f"Factory voltage: {v_before:.4f} -> {fac_bus.vm_pu:.4f} p.u.")

# Back to the base case
factory_load.p_mw = 30.0
grid.solve()
print(f"Restored: {fac_bus.vm_pu:.4f} p.u.")

SolveReport(converged=true, iterations=1, runtime_ms=0.124, rebuild='incremental')
Factory voltage: 1.0135 -> 1.0110 p.u.
Restored: 1.0135 p.u.
